In [ ]:
!pip install -q datasets transformers accelerate evaluate scikit-learn optuna

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import numpy as np
import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
raw_datasets = load_dataset("ailsntua/QEvasion")

model_name = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_df = raw_datasets['train'].to_pandas()

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=700,
    random_state=SEED,
    stratify=train_df['clarity_label']
)

raw_datasets['train'] = Dataset.from_pandas(train_df_split, preserve_index=False)
raw_datasets['validation'] = Dataset.from_pandas(val_df_split, preserve_index=False)

label2id = {
    "Clear Reply": 0,
    "Ambivalent": 1,
    "Clear Non-Reply": 2
}
id2label = {v: k for k, v in label2id.items()}

In [ ]:
def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=3000,
        padding=False
    )
    tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
    return tokenized

tokenized_datasets = {}
tokenized_datasets['train'] = raw_datasets['train'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)
tokenized_datasets['validation'] = raw_datasets['validation'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['validation'].column_names
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

In [ ]:
class EarlyStopping(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True
            
    def reset(self):
        self.best_metric = None
        self.patience_counter = 0

In [ ]:
class OptunaPruningCallback(TrainerCallback):
    def __init__(self, trial, patience: int = 3, metric_name: str = "eval_f1"):
        self.trial = trial
        self.patience = patience
        self.metric_name = metric_name
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name, 0.0)
        epoch = int(state.epoch)
        
        self.trial.report(current_metric, epoch)
        
        if self.trial.should_prune():
            print(f"\nTrial pruned by Optuna at epoch {epoch}")
            raise optuna.TrialPruned()
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if current_metric > self.best_metric:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping at epoch {epoch}!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-6, 1e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.15)
    gradient_accumulation_steps = trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4, 8])
    
    num_train_epochs = 15  
    per_device_train_batch_size = 8  
    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )
    
    training_args = TrainingArguments(
        output_dir=f"./optuna_trials/trial_{trial.number}",
        
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        
        per_device_eval_batch_size=4,
        gradient_checkpointing=True,
        
        eval_strategy="epoch",
        save_strategy="no",  
        load_best_model_at_end=False, 
        metric_for_best_model="f1",
        greater_is_better=True,
        
        logging_dir=f"./optuna_logs/trial_{trial.number}",
        logging_steps=50,
        logging_strategy="steps",
        report_to="none",
        
        fp16=False,
        bf16=True,
        optim="adamw_torch_fused",
        seed=SEED,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[OptunaPruningCallback(trial, patience=3, metric_name="eval_f1")],
    )
    
    try:
        trainer.train()
    except optuna.TrialPruned:
        raise
    
    eval_results = trainer.evaluate()
    best_f1 = eval_results["eval_f1"]
    
    del model
    del trainer
    torch.cuda.empty_cache()
    
    return best_f1

In [ ]:

study = optuna.create_study(
    study_name="modernbert_clarity_optimization",
    direction="maximize",  
    sampler=TPESampler(
        seed=SEED,
        n_startup_trials=5,   
        multivariate=True,    
    ),
    pruner=MedianPruner(
        n_startup_trials=3,   
        n_warmup_steps=2,    
        interval_steps=1,     
    ),
)

In [ ]:
N_TRIALS = 30  

study.optimize(
    objective,
    n_trials=N_TRIALS,
    timeout=None,  
    show_progress_bar=True,
    gc_after_trial=True,  
)

In [ ]:

print("Best trial:")

best_trial = study.best_trial
print(f"  Trial number: {best_trial.number}")
print(f"  Best Validation F1: {best_trial.value:.4f}")

print("\n  Best Hyperparameters:")
for key, value in best_trial.params.items():
    if isinstance(value, float):
        print(f"    {key}: {value:.6f}")
    else:
        print(f"    {key}: {value}")

In [ ]:
best_params = study.best_trial.params

final_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

final_training_args = TrainingArguments(
    output_dir="./results_modernbert_optimized",
    
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    warmup_ratio=best_params["warmup_ratio"],
    gradient_accumulation_steps=best_params["gradient_accumulation_steps"],
    
    num_train_epochs=15,
    per_device_train_batch_size=8,
    
    per_device_eval_batch_size=4,
    gradient_checkpointing=True,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,           
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    logging_dir="./logs_optimized",
    logging_steps=10,
    logging_strategy="steps",
    report_to="none",
    
    fp16=False,
    bf16=True,
    optim="adamw_torch_fused",
    seed=SEED,
)


final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStopping(patience=3, metric_name="eval_f1", greater_is_better=True)],
)

final_trainer.train()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_dataset_raw = load_dataset("ailsntua/QEvasion")['test']

def tokenize_test(examples):
    texts = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples['question'], examples['interview_answer'])
    ]
    return tokenizer(texts, truncation=True, padding=False, max_length=3000)

test_tokenized = test_dataset_raw.map(
    tokenize_test,
    batched=True,
    remove_columns=[col for col in test_dataset_raw.column_names
                    if col not in ['clarity_label']]
)

predictions_output = final_trainer.predict(test_tokenized)
predictions = np.argmax(predictions_output.predictions, axis=-1)
predicted_labels = [id2label[pred] for pred in predictions]

y_true_clarity = [test_dataset_raw[i]['clarity_label'] for i in range(len(test_dataset_raw))]
y_pred_clarity = predicted_labels

accuracy_clarity = accuracy_score(y_true_clarity, y_pred_clarity)
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
macro_f1_clarity = f1_score(y_true_clarity, y_pred_clarity, average='macro', labels=clarity_labels, zero_division=0)

print(f"Accuracy: {accuracy_clarity:.4f}")
print(f"Macro F1: {macro_f1_clarity:.4f}")

print("\nPer-class metrics:")
print(classification_report(y_true_clarity, y_pred_clarity, labels=clarity_labels, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_clarity, y_pred_clarity, labels=clarity_labels)
print(f"Labels order: {clarity_labels}")
print(cm)

In [ ]:
from huggingface_hub import login

login()

repo_name = "gabrielstefan04/modernbert-large-political-evasion"

final_trainer.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)
